# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring a dataset using the `mlcroissant` library, based on the Croissant metadata standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Croissant @id: {meta.id}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")
print(f"Temporal coverage: {meta.temporal_coverage}")


## 2. Data Overview
Review available record sets and their properties, referencing all entities by their `@id`.

In [ ]:
# List all record sets in metadata (using their @id)
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):")

record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    record_set_ids.append(rs.id)
    # List fields for this RecordSet
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for fld in rs.fields:
            print(f"    - Field @id: {fld.id}, Name: {fld.name}")
    print()
# Preview the first RecordSet's records (if any record sets exist)
if record_sets:
    preview_rs = record_sets[0].id
    print(f"Sample record from RecordSet @id: {preview_rs}")
    for i, rec in enumerate(dataset.records(record_set=preview_rs)):
        print(rec)
        if i > 0:
            break
else:
    print("No record sets available in this dataset.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. **All references are by `@id`.**

In [ ]:
# Extract records from all record sets into DataFrames (using their @id)
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecordSet @id: {rs_id}")
    print(f"  DataFrame shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")

# Preview the first few rows of the first DataFrame (if any)
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nPreview of DataFrame for RecordSet @id: {example_rs_id}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA such as filtering, normalizing numeric columns, and simple grouping. All references to fields/columns use their `@id`.

In [ ]:
import numpy as np

# For demonstration, find the first numeric/float/int column to use by @id; otherwise, use default
if record_set_ids:
    df = dataframes[example_rs_id]
    # Try to pick first numeric column (float or int)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print('No numeric field found in the record set.')
    else:
        print(f'Using numeric field by @id: {numeric_field_id}')

        # Define a threshold; pick a quantile if possible
        threshold = np.percentile(df[numeric_field_id].dropna(), 75) if not df[numeric_field_id].dropna().empty else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f'Filtered records with {numeric_field_id} > {threshold:.2f}:')
        display(filtered_df.head())

        # Normalize column
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by the first non-numeric field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'O':
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize distribution of the selected numeric field or relationships with a selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field, if found above
if record_set_ids and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # Boxplot by group field
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we loaded, examined, and visualized the FAIR² dataset using the `mlcroissant` library, referencing all schema entities by their `@id`. With this standardized approach, it's straightforward to extend to more advanced analyses, link to metadata, or automate pipelines using Croissant-compliant datasets.

Key steps included:
- Loading metadata and extracting data with consistent `@id` referencing
- Providing an overview of available record sets and fields
- Filtering and transforming values for preliminary EDA
- Visualizing distributions and relationships between variables

For further analyses, refer to the Croissant schema and data dictionary to interpret each field's meaning and ensure ethical, appropriate use.